In [1]:
import json
from pathlib import Path

nb = json.loads(Path("../urlSplit/extract.ipynb").read_text())
for cell in nb["cells"]:
    if cell["cell_type"] == "code":
        exec("".join(cell["source"]), globals())

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../Data/clean_data.csv")
features = pd.DataFrame([extraction_features(url) for url in df["URL"]])
data = df.join(features)

X = data[NUM_COLS]
y = data["label"]

train_feats, test_feats, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [3]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, f1_score, make_scorer

phish_f1 = make_scorer(f1_score, pos_label=0)

param_grid = {
    "n_estimators":     [200, 300, 500, 800],
    "max_depth":        [3, 4, 6, 8, 10],
    "learning_rate":    [0.01, 0.05, 0.1, 0.2],
    "subsample":        [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma":            [0, 0.1, 0.3],
}

search = RandomizedSearchCV(
    estimator=XGBClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_grid,
    n_iter=30,       
    scoring=phish_f1,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
    verbose=1,
    random_state=42,
)
search.fit(train_feats, y_train)

print("Best settings:", search.best_params_)
print("Best CV phishing F1:", round(search.best_score_, 4))

best_model = search.best_estimator_
pred = best_model.predict(test_feats)
print(classification_report(y_test, pred, target_names=["phishing", "legitimate"]))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best settings: {'subsample': 0.8, 'n_estimators': 800, 'min_child_weight': 5, 'max_depth': 10, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 1.0}
Best CV phishing F1: 0.9022
              precision    recall  f1-score   support

    phishing       0.94      0.87      0.90     20104
  legitimate       0.91      0.96      0.93     26970

    accuracy                           0.92     47074
   macro avg       0.92      0.91      0.92     47074
weighted avg       0.92      0.92      0.92     47074

